```python
from json import loads
import numpy as np

def f():
    global m
    m = {"。": 0}
    for i in "train", "valid":
        for i in open(f"../tmp/news2016zh_{i}.json"):
            i = loads(i)
            for i in i["keywords"], i["desc"], i["title"], i["source"], i["content"]:
                for i in i:
                    yield m.setdefault(i, len(m))
                if "\u4e00" <= i <= "\u9fff":
                    yield 0

a = np.fromiter(f(), "H")
a.tofile("../tmp/news2016zh2.H")
open("../tmp/news2016zh2.txt", "w").write("".join(sorted(m, key=m.__getitem__)))
```

```python
import numpy as np
from multiprocessing.pool import ThreadPool

def f(i):
    c = np.zeros(21482, "I")
    np.add.at(c, a[len(a) * i // 12 : len(a) * (i + 1) // 12], 1)
    return c

a = np.memmap(f"../tmp/news2016zh.H", "H")
with ThreadPool(12) as p:
    p = np.sum(p.map(f, range(12)), 0)

t @ np.log2(t) #-9.655227239498798
    
t = p > p.max() // 100
m = np.zeros(21482, "H")
m[t] = range(t.sum())
m[a[t[a]]].tofile("../tmp/news2016zh100.H")
open("../tmp/news2016zh100.txt", "w").write(np.array([open("../tmp/news2016zh.txt").read()]).view("U1")[t].view(f"U{t.sum()}")[0])

t = p > p.max() // 1000
m = np.zeros(21482, "H")
m[t] = range(t.sum())
m[a[t[a]]].tofile("../tmp/news2016zh1000.H")
open("../tmp/news2016zh1000.txt", "w").write(np.array([open("../tmp/news2016zh.txt").read()]).view("U1")[t].view(f"U{t.sum()}")[0])
```

```python
print(np.array([open("../tmp/news2016zh100.txt").read()]).view("U1")[np.fromfile('../tmp/news2016zh100.H','H',10000)].view('U10000')[0])
```

In [ ]:
from fastai.vision.all import *
from torch.nn import *

torch.cuda.set_device(1)


class Dat:
    a = torch.from_numpy(np.memmap(f"../tmp/news2016zh100.H", "h"))
    m = np.array([open("../tmp/news2016zh100.txt").read()]).view("U1")

    def __init__(s, w, b, n=1):
        s.a = s.a.as_strided([len(s.a) - w + 1, w], [1, 1])
        s.b = b
        s.n = n

    def __len__(s):
        return len(s.a) // (s.a.size(1) * s.b * s.n)

    def __iter__(s):
        for x in range(len(s)):
            x = s.a[torch.randint(len(s.a), [s.b])].to("cuda", torch.long)
            yield x, x


class View(Module):
    def __init__(s, k):
        super().__init__()
        s.k = k

    def forward(s, x):
        return x.view(*x.shape[:-2], -1, s.k)


class PyramidAutoEncoder(Sequential):
    def forward(s, x):
        n = int(np.log2(x.size(1)))
        t = []
        for _ in range(n):
            x = s[0](x)
            t.append(x)
        t = list(s[1](torch.cat(t, 1)).split([t.size(1) for t in t], 1))
        x = None
        while t:
            x = s[2](t.pop()[None] if x is None else torch.cat([t.pop()[None], x]))
        return x


class MultiCrossEntropyLoss(CrossEntropyLoss):
    def forward(s, x, t):
        x = x.view(len(x), -1, x.size(-1))
        return -x.log_softmax(2)[:, torch.arange(x.size(1), device="cuda"), t.flatten()
        ].mean()


def Accuracy(i):
    def Accuracy(x, t):
        return accuracy(x[i], t)

    Accuracy.__name__ = str(int(i))
    return Accuracy

In [ ]:
m = Learner(
    DataLoaders(Dat(256, 256, 10), Dat(256, 256, 1000)),
    Sequential(
        Embedding(560, 24),
        PyramidAutoEncoder(
            Sequential(View(48), Linear(48, 24, 0), BatchNorm1dFlat(24, track_running_stats=0), Swish()),
            Sequential(Linear(24, 24, 0), BatchNorm1dFlat(24, track_running_stats=0), Swish(), Linear(24, 24)),
            Sequential(BatchNorm1dFlat(24, track_running_stats=0), Swish(), Linear(24, 48, 0), View(24)),
        ),
        Linear(24, 24, 0),
        BatchNorm1dFlat(24, track_running_stats=0),
        Swish(),
        Linear(24, 560),
    ).cuda(),
    MultiCrossEntropyLoss(),
    metrics=map(Accuracy, range(8)),
    cbs=[ShowGraphCallback, SaveModelCallback(fname='model2b', every_epoch=1, with_opt=1, reset_on_fit=0)],
)
m.load('model2', 'cuda')
# m.lr_find()

In [ ]:
x = next(iter(m.dls[1]))[0]
print(m.dls.m[x[0].cpu()].view('U256')[0])

In [ ]:
for i in m.dls.m[m.model(x)[:,0].argmax(2).cpu()].view('U256')[:,0]:
    print(i, end='\n====================\n')